In [7]:
!pip -q install dataset

In [35]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from datasets import load_dataset
from collections import Counter

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 100)

Load the FiQA data

In [9]:
dataset = load_dataset("LLukas22/fiqa")
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 14511
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 2561
    })
})

In [10]:
#print(dataset)
for split_name, split_data in dataset.items():
    print(f"{split_name}: {len(split_data):,} rows")

train: 14,511 rows
test: 2,561 rows


Then combine all original splits into a single dataframe. We are doing this because we will later create our own split after filtering, and we want to prevent related questions from leaking across train/validation/test.

In [11]:
frames = []

for split_name, split_data in dataset.items():
    temp = split_data.to_pandas()
    temp["original_split"] = split_name
    frames.append(temp)

df = pd.concat(frames, ignore_index=True)

print("Shape:", df.shape)
display(df.head())

Shape: (17072, 3)


,question,answer,original_split
0,What do brokers do with bad stock?,"For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, foolish belief in a crazy business, etc). The party (brokerage, market maker, individual) owning the stock at the time the company goes out of business is the loser . But in a general panic, not ...",train
1,Why do investors buy stock that had appreciated?,"You seem to prefer to trade like I do: ""Buy low, sell high."" But there are some people that prefer a different way: ""Buy high, sell higher."" A stock that has ""just appreciated"" is ""in motion."" That is a ""promise"" (not always kept) that it will continue to go higher. Some people want stocks that ...",train
2,Is it ever a good idea to close credit cards?,"Yes, it can be a good idea to close unused credit cards. I am going to give some reasons why it can be a good idea to close unused accounts, and then I will talk about why it is NOT necessarily a bad idea. Why it can be a good idea to close unused accounts ""I'd like to close the cards."" That i...",train
3,If something is coming into my account will it be debit or credit in my account?,The bank will make this even more confusing because they use the terms from their own perspective. From the bank's perspective (printed on your statements) credit: Money into your account (increases the bank's liabilities) debit: Money out of your account (decrease bank liabilities) From your...,train
4,What one bit of financial advice do you wish you could've given yourself five years ago?,When I was contracting I wish I had joined a tax efficient umbrella organisation rather than just work as a sole trader. I also wish I had put money aside to pay my taxes rather than just spend it all. :(,train


Let's now do a basic structural Audit

In [12]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nUnique values:")
print("Unique questions:", df["question"].nunique())
print("Unique answers:", df["answer"].nunique())

print("\nDuplicate QA pairs:", df.duplicated(subset=["question", "answer"]).sum())

print("\nDuplicated questions:", df.duplicated(subset=["question"]).sum())

Dataset shape: (17072, 3)

Columns:
['question', 'answer', 'original_split']

Data types:
question          object
answer            object
original_split    object
dtype: object

Missing values:
question          0
answer            0
original_split    0
dtype: int64

Unique values:
Unique questions: 6646
Unique answers: 17070

Duplicate QA pairs: 0

Duplicated questions: 10426


In [13]:
display(
    df.sample(
        min(10, len(df)),
        random_state=42
    )[["question", "answer", "original_split"]]
)

,question,answer,original_split
6015,Will there always be somebody selling/buying in every stock?,"Many people assume that if the price of something is $10 and they have 1,000 of that thing, they should expect to be able to sell them for something around $10,000. Such an assumption may hold much of the time, but it doesn't always. Worse, the cases where it fails to hold are often those wher...",train
1147,Some stock's prices don't fluctuate widely - Is it an advantages?,"What is your investment goal? Many investors buy for the long haul, not short-term gain. If you're looking for long-term gain then daily fluctuations should be of no concern to you. If you want to day-trade and time the market (buy low and sell high with a short holding period) then yes less ...",train
5149,Including the region where you live in your investment portfolio?,"Diversification is a risk-mitigation strategy. When you invest in equities, you generally get a higher rate of return than a fixed income investment. But you have risks... a single company's market value can decline for all sorts of reasons, including factors outside of the control of management...",train
8762,What are my investment options in real estate?,"I compared investing in real estate a few years ago to investing in stocks that paid double digit dividends (hard to find, however, managing and maintaining real estate is just as hard). After discussing with many in the real estate world, I counted the average and learned that most averaged ab...",train
5886,What happens if stock purchased on margin plummets below what I have in the brokerage?,"Different brokerages have different house rules for margin requirements and margin calls. You will likely get a margin call giving you a small amount of time to deposit the required funds to bring your account balance up to the required margin requirements. In reality, a stock that falls from ...",train
1086,What differentiates index funds and ETFs?,"I think that assuming that you're not looking to trade the fund, an index Mutual Fund is a better overall value than an ETF. The cost difference is negligible, and the ability to dollar-cost average future contributions with no transaction costs. You also have to be careful with ETFs; the spread...",train
6069,Beginner dividend investor - first steps,"Question 1: How do I start? or ""the broker"" problem Get an online broker. You can do a wire transfer to fund the account from your bank. Question 2: What criticism do you have for my plan? Dividend investing is smart. The only problem is that everyone's currently doing it. There is an insatia...",train
12349,Austrailian tax resident earning salary in the UK - how much tax do I pay on foreign income?,"This page and this page on the ATO website provide some information on tax rates. They're rather lengthy and there's a few exceptions, but essentially, your entire foreign income, even if held overseas, is taxable. Australians are taxed worldwide.",train
13758,How should I save money if the real interest rate (after inflation) is negative?,"Inflation protected securities (i-bonds or TIPS). TIPS stands for Treasury Inflation Protected Securities. By very definition, they tend to protect your savings against inflation. They won't beat inflation, but will keep up with it. TIPS or iBonds have two parts. A fixed interest part and a va...",train
13569,Why could rental costs for apartments/houses rise while buying prices can go up and down?,"Average rent rates will typically rise and fall, and are market-dependent just like real estate. In the short term, a collapse in housing like the one we saw in 2008 can induce a spike in rental costs as people walk away or get foreclosed on, and move back into apartments. That then tends to sel...",train


This tells something important: FiQA is broad. It contains some highly relevant personal-finance questions such as saving money, credit cards, index funds, beginner investing, and account debit/credit, but also plenty of less relevant material such as foreign taxation, margin trading, and advanced stock-market questions.
<br>That's why we will do a careful relevance-filtering stage rather than treating all 17k rows as suitable.

Let's now inspect question and answer lengths

In [14]:
df["question_chars"] = df["question"].astype(str).str.len()
df["answer_chars"] = df["answer"].astype(str).str.len()

df["question_words"] = df["question"].astype(str).str.split().str.len()
df["answer_words"] = df["answer"].astype(str).str.split().str.len()

length_summary = df[["question_chars", "answer_chars", "question_words", "answer_words"]
                    ].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])

display(length_summary)

,question_chars,answer_chars,question_words,answer_words
count,17072.000000,17072.000000,17072.000000,17072.000000
mean,62.531338,1032.588683,11.126640,183.323102
std,22.645344,914.826668,4.387252,161.828679
min,14.000000,6.000000,2.000000,1.000000
50%,61.000000,784.000000,11.000000,140.000000
75%,77.000000,1321.000000,14.000000,235.000000
90%,92.000000,2089.900000,17.000000,372.000000
95%,103.000000,2741.250000,19.000000,483.000000
99%,127.000000,4441.290000,23.000000,783.580000
max,166.000000,16984.000000,31.000000,2973.000000


Let's display the 5 longest answers and then the 10 shortest answer

In [15]:
display(df.nlargest(5, "answer_words")[
        ["question", "answer", "answer_words"]])

,question,answer,answer_words
10149,How can I pay for school to finish my degree when I can't get a student loan and have bad credit?,"When considering such a major life decision, with such high potential costs and high potential rewards, I encourage you to consider multiple different potential options. Even if loans were available, they might not be the best option. Less debt and an engineering degree is better than more debt ...",2973
10479,What happens to people without any retirement savings?,"I'm afraid you have missed a few of the outcomes commonly faced by millions of Americans, so I would like to take a moment to discuss a wider range of outcomes that are common in the United States today. Most importantly, some of these happen before retirement is ever reached, and have grave con...",2189
12485,Using pivot points to trade in the short term,"Pivots Points are significant levels technical analysts can use to determine directional movement, support and resistance. Pivot Points use the prior period's high, low and close to formulate future support and resistance. In this regard, Pivot Points are predictive or leading indicators. There ...",1945
8165,Why do 10 year Treasury bond yields affect mortgage interest rates?,You’ve really got three or four questions going here… and it’s clear that a gap in understanding one component of how bonds work (pricing) is having a ripple effect across the other facets of your question. The reality is that everybody’s answers so far touch on various pieces of your general qu...,1882
13652,"What exactly is a ""derivative""?","A derivative is a financial instrument of a special kind, the kind ""whose price depends on, or is derived from, another asset"". This definition is from John Hull, Options, Futures and Other Derivatives – a book definitely worth to own if you are curious about this, you can easily find old copie...",1762


In [16]:
display(df.nsmallest(10, "answer_words")[
        ["question", "answer", "answer_words"]])

,question,answer,answer_words
6438,Difference between GOOGL and GOOG,Source,1
8699,What's my risk of buying a house for a friend and sell back to him?,Risks:,1
4324,How to take advantage of record high household debt in Canada?,Some ideas:,2
6086,What are the marks of poor investment advice?,Bad signs:,2
6764,Ongoing things to do and read to improve knowledge of finance?,Good luck!,2
6794,What one bit of financial advice do you wish you could've given yourself five years ago?,2 things:,2
9655,What economic growth rate is required to halve U.S. unemployment?,Two points.,2
998,What are the best software tools for personal finance?,Excel Pros: Cons:,3
10229,What are the best software tools for personal finance?,For iPhone: iExpenseIt,3
10897,Borrow from 401k for down payment on rental property?,Two simple possibilities:,3


are clearly not useful standalone QA targets. These appear to be incomplete fragments, probably caused by preprocessing of the original source content or formatting artifacts. Let's investigate them further.

In [17]:
short_thresholds = [1, 2, 3, 5, 10, 15, 20]

for threshold in short_thresholds:
    count = (df["answer_words"] <= threshold).sum()
    pct = count / len(df) * 100

    print(f"Answers <= {threshold:2d} words: {count:4d} ({pct:.2f}%)")

Answers <=  1 words:    2 (0.01%)
Answers <=  2 words:    7 (0.04%)
Answers <=  3 words:   13 (0.08%)
Answers <=  5 words:   31 (0.18%)
Answers <= 10 words:  116 (0.68%)
Answers <= 15 words:  244 (1.43%)
Answers <= 20 words:  422 (2.47%)


In [18]:
very_short = df[df["answer_words"] <= 5][
    ["question", "answer", "answer_words"]]

print("Number of answers with <= 5 words:", len(very_short))
display(very_short)

Number of answers with <= 5 words: 31


,question,answer,answer_words
847,ETF vs Mutual Fund: How to decide which to use for investing in a popular index?,The factors to consider:,4
998,What are the best software tools for personal finance?,Excel Pros: Cons:,3
4324,How to take advantage of record high household debt in Canada?,Some ideas:,2
4463,When is Cash Value Life Insurance a good or bad idea?,Here's what I'd consider:,4
6086,What are the marks of poor investment advice?,Bad signs:,2
6438,Difference between GOOGL and GOOG,Source,1
6764,Ongoing things to do and read to improve knowledge of finance?,Good luck!,2
6794,What one bit of financial advice do you wish you could've given yourself five years ago?,2 things:,2
7832,Dad paying for my new home in cash. How can I buy the house from him?,You have four basic options.,5
8134,What are the best software tools for personal finance?,Money Manager Ex PROS: CONS,5


In [19]:
short_6_10 = df[(df["answer_words"] >= 6) & (df["answer_words"] <= 10)][["question", "answer", "answer_words"]]

print("Answers between 6 and 10 words:", len(short_6_10))

display(short_6_10.sample(min(30, len(short_6_10)),random_state=42))

Answers between 6 and 10 words: 85


,question,answer,answer_words
16840,What are some time tested passive income streams?,Any kind of savings account is a passive income stream.,10
268,Would you withdraw your money from your bank if you thought it was going under?,I have two different thoughts on this subject.,8
15293,Can I buy only 4 shares of a company?,Yes you can. it's called Odd Lot,7
5705,"Someone asks you to co-sign a loan. How to reject & say ""no"" nicely or politely?","No, I don't mix business and personal affairs.",8
3150,Where can I borrow money for investing?,Have you considered social lending (for example: Lending Club)?,9
17025,How do I get a list of the top performing funds between two given dates?,I found one such tool here: Point-to-Point Returns tool,9
2314,What are some time tested passive income streams?,Royalties. (Once you start getting them.),6
4755,"If I make over 120k a year, what are my options for retirement plans?",There are three common options for you:,7
920,Historical company performance data,Morningstar has that 10 history at http://financials.morningstar.com/ratios/r.html?t=JNJ&region=usa&culture=en-US,7
15220,Asking price went through the roof,As folks have explained in the comments:,7


This gives us enough evidence to make a defensible cleaning decision. **The ≤5-word** answers should be removed. There are only **31 rows out of 17,072 (~0.18%)**, and our inspection shows that almost all of them are headings, incomplete fragments, bare links, or extremely low-information responses.

By contrast, we would not remove the 6–10 word answers automatically. The sample above contains several perfectly legitimate concise responses, such as "The balance is the amount due."

In [20]:
MIN_ANSWER_WORDS = 6

In [21]:
answers_per_question = (
    df.groupby("question")
      .size()
      .sort_values(ascending=False)
)

print("Unique questions:", len(answers_per_question))

display(answers_per_question.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nQuestions with multiple answers:", (answers_per_question > 1).sum())

print("\nMaximum answers for one question:", answers_per_question.max())

Unique questions: 6646


,0
count,6646.000000
mean,2.568763
std,2.139249
min,1.000000
50%,2.000000
75%,3.000000
90%,5.000000
95%,7.000000
99%,11.000000
max,23.000000



Questions with multiple answers: 4201

Maximum answers for one question: 23


In [22]:
# Let's inspect the five questions with the most answers
top_questions = answers_per_question.head(5).index.tolist()

for q in top_questions:
    print("=" * 70)
    print("QUESTION:", q)

    subset = df[df["question"] == q]

    print(f"\nNumber of answers: {len(subset)}")

QUESTION: What options do I have at 26 years old, with 1.2 million USD?

Number of answers: 23
QUESTION: Why buy insurance?

Number of answers: 23
QUESTION: Why would you ever turn down a raise in salary?

Number of answers: 22
QUESTION: If a stock doesn't pay dividends, then why is the stock worth anything?

Number of answers: 21
QUESTION: Who can truly afford luxury cars?

Number of answers: 21


In [23]:
answer_count_distribution = (
    answers_per_question
    .value_counts()
    .sort_index()
    .rename_axis("number_of_answers")
    .reset_index(name="number_of_questions")
)

display(answer_count_distribution)

,number_of_answers,number_of_questions
0,1,2445
1,2,1794
2,3,1044
3,4,567
4,5,279
5,6,177
6,7,110
7,8,64
8,9,53
9,10,39


Let's now move on to the actual cleaning stage.

In [24]:
def clean_text(text):
    if pd.isna(text):
        return np.nan

    text = str(text)
    text = re.sub(r"\s+", " ", text) # Normalize whitespace
    text = text.strip()  # Remove leading/trailing spaces

    return text

df_clean = df.copy()
df_clean["question_clean"] = df_clean["question"].apply(clean_text)
df_clean["answer_clean"] = df_clean["answer"].apply(clean_text)
print("Initial rows:", len(df_clean))

Initial rows: 17072


First, we remove missing/empty records

In [25]:
df_clean = df_clean.dropna(
    subset=["question_clean", "answer_clean"]).copy()

df_clean = df_clean[
    (df_clean["question_clean"].str.len() > 0) &
    (df_clean["answer_clean"].str.len() > 0)].copy()

print("After missing/empty removal:", len(df_clean))

After missing/empty removal: 17072


In [26]:
df_clean["answer_words"] = (df_clean["answer_clean"].str.split().str.len())

MIN_ANSWER_WORDS = 6

short_removed = df_clean[
    df_clean["answer_words"] < MIN_ANSWER_WORDS].copy()

df_clean = df_clean[
    df_clean["answer_words"] >= MIN_ANSWER_WORDS].copy()

print("Short answers removed:", len(short_removed))
print("Rows remaining:", len(df_clean))

Short answers removed: 31
Rows remaining: 17041


In [27]:
duplicate_count = df_clean.duplicated(
    subset=["question_clean", "answer_clean"]).sum()

print("Exact duplicate QA pairs after cleaning:", duplicate_count)

Exact duplicate QA pairs after cleaning: 0


In [29]:
incomplete_colon = df_clean[
    df_clean["answer_clean"].str.endswith(":")]

print("Answers ending with a colon:", len(incomplete_colon))

display(incomplete_colon[
    ["question_clean", "answer_clean", "answer_words"]].head(15))

Answers ending with a colon: 354


,question_clean,answer_clean,answer_words
83,Accepting high volatility for high long-term returns,"Modern portfolio theory dramatically underestimates the risk of the recommended assets. This is because so few underlying assets are in the recommended part of the curve. As investors identify such assets, large amounts of money are invested in them. This temporarily reduces measured risk, and t...",104
222,How to get into real estate with a limited budget,"You are neglecting a few very important things around real estate transactions in Belgium So in the end a 300K building may cost you more than 340K, let's take some unexpected costs into account and use 350K for remainder of calculation. Even worse if it's newly built (which I doubt) the first p...",624
234,"What exactly is a ""derivative""?",There are a few unsavory factors that have led to the creation of new derivatives:,15
255,Does borrowing from my 401(k) make sense in my specific circumstance?,"I completely agree with Pete that a 401(k) loan is not the answer, but I have an alternate proposal: Reduce your 401(k) contribution down to the 4% that you get a match on. If you are cash poor now and have debts to be cleaned up, those need to be addressed before retirement savings. You'll have...",382
321,"AVS Address Verification System of BOTH Credit and Debit Cards - WHERE, HOW?","Parts of what you want are possible, but taken as a whole, you're out of luck. First of all, there is no master database of every cardholder in the country. The only way to check if information is correct is to ask the issuing bank. The AVS system is a way to automate doing so, but it's possible...",215
325,Is the interest on money borrowed on margin in/for an RRSP considered tax deductible?,"I believe your question is based on a false premise. First, no broker, that I know of, provides an RRSP account that is a margin account. RRSP accounts follow cash settlement rules. If you don't have the cash available, you can't buy a stock. You can't borrow money from your broker within your R...",162
343,What industries soar when oil prices go up?,You can look at it from a fundamental perspective to see who benefits from rising oil prices. That's a high level analysis and the devil is in the details - higher oil prices may favour electric car producers for example or discount clothes retailers vs. branded clothes manufacturers. Another ap...,213
434,How to pay bills for one month while waiting for new job?,"This is just a partial answer, but I believe the following observations are relevant:",14
616,To rebalance or not to rebalance,"Yes E[x] is expected value of x. E[x|y] = expected value of x, given y. c, k are some constants Let E[s_{n+1}|s_n=c] = c, but if E[s_{n+1}|s_n,s_{n-1},...,s_{n-m}] ->some constant k as m->\infty (call this equation 1) then rebalancing makes sense. Notes:",41
680,Rent or buy with 0 down,"In the situation you describe, I would strongly consider purchasing. Before purchasing, I would do the following: Think about your goals. Work with good people. Set a budget. Be able to handle surprises. If buying a home makes sense, you can do the following after buying:",46


The output above indicates that we should not automatically remove answers just because they end with a colon. Let's just flag them and we will revisit them later.

In [30]:
df_clean["ends_with_colon"] = (
    df_clean["answer_clean"]
    .str.strip()
    .str.endswith(":")
)

## Domain filtering

First, let's define the target taxonomy

In [31]:
TARGET_TOPICS = {
    "budgeting_spending": [
        "budget",
        "budgeting",
        "spending",
        "spend",
        "expense",
        "expenses",
        "living expenses",
        "cost of living",
        "afford",
        "affordable",
        "living within",
        "needs and wants"
    ],

    "saving_emergency_funds": [
        "save",
        "saving",
        "savings",
        "emergency fund",
        "rainy day fund",
        "financial goal",
        "financial goals"
    ],

    "student_finance": [
        "student loan",
        "student loans",
        "student debt",
        "college",
        "university",
        "tuition",
        "student",
        "scholarship",
        "financial aid",
        "school loan"
    ],

    "banking": [
        "bank account",
        "savings account",
        "checking account",
        "current account",
        "banking",
        "bank",
        "deposit",
        "withdrawal",
        "overdraft",
        "debit card"
    ],

    "credit_and_debt": [
        "credit card",
        "credit cards",
        "credit score",
        "credit history",
        "credit report",
        "debt",
        "loan",
        "loans",
        "borrow",
        "borrowing",
        "interest rate",
        "repay",
        "repayment"
    ],

    "income_and_cashflow": [
        "income",
        "salary",
        "paycheck",
        "pay cheque",
        "wage",
        "wages",
        "allowance",
        "cash flow",
        "cashflow",
        "monthly income",
        "side income"
    ],

    "basic_investing": [
        "invest",
        "investing",
        "investment",
        "index fund",
        "index funds",
        "mutual fund",
        "mutual funds",
        "etf",
        "compound interest",
        "diversification",
        "portfolio",
        "retirement savings"
    ],

    "financial_planning": [
        "financial planning",
        "financial plan",
        "personal finance",
        "net worth",
        "financial goal",
        "financial goals",
        "financial advice",
        "manage money",
        "money management"
    ],

    "insurance_and_risk": [
        "insurance",
        "insured",
        "insure",
        "health insurance",
        "life insurance",
        "risk management"
    ],

    "financial_safety": [
        "scam",
        "scams",
        "fraud",
        "identity theft",
        "financial scam",
        "phishing"
    ]
}

Next, let's search primary in the question

In [32]:
def contains_keyword(text, keyword):
    """
    Match keywords as words/phrases rather than arbitrary substrings.
    """
    pattern = r"\b" + re.escape(keyword.lower()) + r"\b"
    return bool(re.search(pattern, text.lower()))


def identify_topics(question):
    matched_topics = []

    for topic, keywords in TARGET_TOPICS.items():
        if any(
            contains_keyword(question, keyword)
            for keyword in keywords
        ):
            matched_topics.append(topic)

    return matched_topics

In [33]:
df_clean["matched_topics"] = (
    df_clean["question_clean"]
    .apply(identify_topics)
)

df_clean["num_topic_matches"] = (
    df_clean["matched_topics"]
    .str.len()
)

In [34]:
# Let's create the candidate set
df_candidates = df_clean[df_clean["num_topic_matches"] > 0].copy()

print("Cleaned FiQA QA pairs:", len(df_clean))
print("Initial relevant candidates:", len(df_candidates))

print("Candidate percentage:", round(len(df_candidates) / len(df_clean) * 100, 2), "%")

Cleaned FiQA QA pairs: 17041
Initial relevant candidates: 6556
Candidate percentage: 38.47 %


Let's count how many QA pairs in each choosen topic (topics of our domain)

In [36]:
topic_counts = Counter(
    topic
    for topics in df_candidates["matched_topics"]
    for topic in topics
)

topic_counts_df = (
    pd.DataFrame(topic_counts.items(), columns=["topic", "qa_pairs"]
    ).sort_values("qa_pairs", ascending=False
    ).reset_index(drop=True)
)

display(topic_counts_df)

,topic,qa_pairs
0,basic_investing,2209
1,credit_and_debt,1952
2,banking,748
3,income_and_cashflow,657
4,saving_emergency_funds,632
5,budgeting_spending,500
6,student_finance,360
7,insurance_and_risk,339
8,financial_safety,187
9,financial_planning,174


But because one question can have multiple answers, we also want to know unique-question counts.

In [37]:
unique_topic_counts = []

for topic in TARGET_TOPICS.keys():

    subset = df_candidates[df_candidates["matched_topics"].apply(lambda topics: topic in topics)]

    unique_topic_counts.append({
        "topic": topic,
        "qa_pairs": len(subset),
        "unique_questions":
            subset["question_clean"].nunique()
    })

topic_summary = (
    pd.DataFrame(unique_topic_counts)
    .sort_values(
        "unique_questions",
        ascending=False
    ).reset_index(drop=True)
)

display(topic_summary)

,topic,qa_pairs,unique_questions
0,basic_investing,2209,797
1,credit_and_debt,1952,611
2,banking,748,272
3,income_and_cashflow,657,271
4,saving_emergency_funds,632,189
5,budgeting_spending,500,159
6,student_finance,360,111
7,insurance_and_risk,339,108
8,financial_planning,174,49
9,financial_safety,187,44


In [40]:
candidate_sample = df_candidates.sample(min(40, len(df_candidates)), random_state=42)[
    [
        "question_clean",
        "matched_topics",
        "answer_clean"
    ]]

display(candidate_sample.head(10))

,question_clean,matched_topics,answer_clean
14510,Should I Purchase Health Insurance Through My S-Corp,[insurance_and_risk],"The answer seems to depend on where you live. Perhaps you already found this, but the summary from the IRS is: The insurance laws in some states do not allow a corporation to purchase group health insurance when the corporation only has one employee. Therefore, if the shareholder was the sole co..."
14727,Should I close unused credit cards before applying for another?,[credit_and_debt],"You want to have 2-4 credit cards, with a credit utilization ratio below 30%. If you only have 2 cards, closing 1 would reduce your credit diversity and thus lower your credit score. You also want at least 2 years credit history, so closing an older credit card may shorten your credit history, a..."
12579,Is this follow-up after a car crash a potential scam?,[financial_safety],"I would write them a check or give them cash money. There are payment receipt forms available online, you can print one of have them fill it out and sign it. Just google ""private party receipt"". Money transfer (via bank account or Paypal) is also an option, but in my opinion it's more convenient..."
11847,I carelessly invested in a stock on a spike near the peak price. How can I salvage my investment?,[basic_investing],"Yes, you could sell what you have and bet against others that the stock price will continue to fall within a period of time ""Shorting"". If you're right, your value goes UP even though the stock price goes down. This is a pretty darn risky bet to make. If you're wrong, there's no limit to how muc..."
14350,How to file income tax returns for profits from ESPP stock?,[income_and_cashflow],I did this for the last tax year so hopefully I can help you. You should get a 1099-B (around the same time you're getting your W-2(s)) from the trustee (whichever company facilitates the ESPP) that has all the information you need to file. You'll fill out a Schedule D and (probably) a Form 8949...
10685,"Why do credit card transactions take up to 3 days to appear, yet debit transactions are instant?",[credit_and_debt],"When you swipe your credit card, the terminal at the store makes a request of your bank, and your bank has only a few seconds to accept or reject the transaction. Once the transaction is accepted by your bank, it appears in the Pending transactions. At the end of the business day, the store subm..."
1588,Retirement savings vs building lucrative assets,"[saving_emergency_funds, basic_investing]","Well... (in the US, at least) ""making investments and building assets"" is how you save for retirement. The investments just happen to be in the stock market, and the federal legislature has directed the US version of Inland Revenue Services to give special tax breaks to investments which are not..."
16763,"Best way to start investing, for a young person just starting their career?",[basic_investing],"Not 100% related, but the #1 thing you need to avoid is CREDIT CARD DEBT. Trust me on this one. I'm 31, and finally got out of credit card debt about eight months ago. For just about my entire 20s, I racked up credit card debt and saved zero. Invested zero. It pains me to realize that I basicall..."
258,Investment / Savings advice in uncertain economy,"[saving_emergency_funds, basic_investing]","$23,000 Student Loans at 4% This represents guaranteed loss. Paying this off quickly is a conservative move, while your other investments may easily surpass 4% return, they are not guaranteed. Should I just keep my money in my savings account since I want to keep my money available? Or are there..."
6255,Is Cost of Living overstated?,[budgeting_spending],"You mention: High rent places are usually also high property value places. Given the tax incentives, it seems like a good long term idea to grab a house, so if we assume you have the option of working and buying a house in a high CoL or a low CoL city, I think you'd prefer the high cost. Because..."


Next, we're going to do question-level exclusion filtering

In [64]:
EXCLUSION_KEYWORDS = {
    "advanced_trading": [
        "short selling",
        "shorting",
        "margin trading",
        "margin call",
        "technical analysis",
        "pivot point",
        "futures",
        "futures contract",
        "derivative",
        "derivatives",
        "forex trading",
        "day trading",
        "intraday",
        "put option",
        "call option",
        "stock options",
        "employee stock options"
    ],

    "corporate_finance": [
        "corporation",
        "s-corp",
        "c-corp",
        "corporate",
        "shareholder",
        "stockholder",
        "ebitda",
        "business expense",
        "business loan"
    ],

    "tax_specific": [
        "tax return",
        "tax returns",
        "tax deduction",
        "tax deductible",
        "income tax",
        "capital gains tax",
        "irs",
        "1099",
        "w-2",
        "w2",
        "schedule c",
        "schedule d",
        "1040",
        "tax year",
        "tax credit"
    ],

    "specific_securities": [
        "stock price",
        "share price",
        "ticker",
        "dividend yield",
        "espp",
        "wash sale",
        "stock exchange",
        "ipo"
    ],

    "business_accounting": [
        "bookkeeping",
        "accounting",
        "depreciation",
        "amortization",
        "owner's earnings",
        "free cash flow",
        "financial statement"
    ]
}

Then create a function to flag these records.

In [65]:
def identify_exclusions(question):
    matched = []

    for category, keywords in EXCLUSION_KEYWORDS.items():
        if any(
            contains_keyword(question, keyword)
            for keyword in keywords
        ):
            matched.append(category)

    return matched

df_candidates["exclusion_flags"] = (df_candidates["question_clean"].apply(identify_exclusions))
df_candidates["num_exclusion_flags"] = (df_candidates["exclusion_flags"].str.len())

Let's now inspect how many got flagged

In [66]:
print("Total candidate QA pairs:", len(df_candidates))

print("QA pairs with at least one exclusion flag:", (df_candidates["num_exclusion_flags"] > 0).sum())

print("Unique questions flagged:",
    df_candidates.loc[df_candidates["num_exclusion_flags"] > 0, "question_clean"].nunique())

Total candidate QA pairs: 6556
QA pairs with at least one exclusion flag: 249
Unique questions flagged: 140


In [67]:
exclusion_counts = Counter(
    flag
    for flags in df_candidates["exclusion_flags"]
    for flag in flags
)

exclusion_summary = (
    pd.DataFrame( exclusion_counts.items(), columns=["exclusion_category", "qa_pairs"])
    .sort_values("qa_pairs", ascending=False)
    .reset_index(drop=True)
)

display(exclusion_summary)

,exclusion_category,qa_pairs
0,tax_specific,132
1,specific_securities,42
2,corporate_finance,37
3,advanced_trading,28
4,business_accounting,20


Let's now create the exclusion rules

In [68]:
df_flagged = df_candidates[df_candidates["num_exclusion_flags"] > 0].copy()

df_unflagged = df_candidates[df_candidates["num_exclusion_flags"] == 0].copy()

print("Unflagged candidate QA pairs:", len(df_unflagged))
print("Flagged candidate QA pairs:", len(df_flagged))

Unflagged candidate QA pairs: 6307
Flagged candidate QA pairs: 249


In [74]:
# Sample the flagged questions
display(
    df_flagged[
        [
            "question_clean",
            "matched_topics",
            "exclusion_flags",
            "answer_clean"
        ]
    ]
    .drop_duplicates("question_clean")
    .sample(min(10, df_flagged["question_clean"].nunique()), random_state=42)
)

,question_clean,matched_topics,exclusion_flags,answer_clean
12427,"When entering a Futures contract, must the margin deposit be idle or can I profit from it?",[banking],[advanced_trading],"In theory, an FCM may accept various types of collateral, including assets such as cash, treasuries, certain stocks, sovereign debt, letters of credit, and (as of 2009, I think,) gold. In practice, most will want you to post cash or cash. Some will take treasuries, but I think you'll generally h..."
5221,How do I report this cash bonus/tip on income tax return?,[income_and_cashflow],[tax_specific],"How do I report this on our income tax return? You should include it on Line 7 of your Form 1040. Additionally, you should report the extra payment to your employer if it was greater that $20. You can use From 4070 to do this if your employer does not provide you with a form. And finally, you ar..."
2464,Should withheld income tax be included as income?,[income_and_cashflow],[tax_specific],This very topic was the subject of a question on workplace SE https://workplace.stackexchange.com/questions/8996/what-can-relocation-assistance-entail TL/DR; From tax publication 521 - Moving expenses table regarding how to report IF your Form W-2 shows... your entire reimbursement reported as w...
13485,Employer reported ESPP ordinary income on wrong year's W-2,[income_and_cashflow],"[tax_specific, specific_securities]","You mentioned that the 1099B that reports this sale is for 2014, which means that you got the proceeds in 2014. What I suspect happened was that the employer reported this on the next available paycheck, thus reporting it in the 2015 period. If this ends up being a significant difference for you..."
3309,Live in Florida & work remote for a New York company. Do I owe NY state income tax?,[income_and_cashflow],[tax_specific],"This question came up again (Living in Florida working remotely - NY employer withholds NYS taxes - Correct or Incorrect?) and the poster on the new version didn't find the existing answers to be adequate, so I'm adding a new answer. NYS will tax this income if the arrangement is for the conveni..."
757,"What are the common income tax deductions used by ""rich"" salaried households?",[income_and_cashflow],[tax_specific],"One of the main tax loopholes more readily available to the wealthy in the U.S. is the fact that long-term capital gains are taxed at a much lower rate. Certainly, people making less than $250,000/year can take advantage of this as well, but the fact is that people making, say, $60,000/year like..."
7629,How to file income tax returns for profits from ESPP stock?,[income_and_cashflow],"[tax_specific, specific_securities]","Consult a professional CA. For shares sold outside the Indian Stock Exchanges, these will be treated as normal Long Term Capital Gains if held more than one year. The rate would be 10% without Indexation and 20% with Indexation. If the stocks are held for less than 1 years, it will be short term..."
5472,"If I pay someone else's property taxes, can I use it as a deduction on my income tax return?",[income_and_cashflow],[tax_specific],"To make matters worse, if you pay the property tax your mother in law can't take the deduction either. You may be better off paying rent and having her handle the property correctly, as a rental."
11279,Can vet / veterinary bills be considered deductions (tax-deductible) for Income Tax purposes [Canada]?,[income_and_cashflow],[tax_specific],"No. Medical bills for yourself or your human companions may be: Canada Revenue Seeing-eye dogs and the like also get special treatment Nice Doggie There are pet medical insurance policies; but as they are often priced like human policies, they might exclude your animal if it has a pre-existing c..."
12492,Strategy for accounting personal finance in multiple currency?,[financial_planning],[business_accounting],"How can I correctly account for having money in different currencies, without currency transfers or currency fluc

Now let's apply the actual exclusion logic

In [75]:
CORE_TOPICS = {
    "student_finance",
    "budgeting_spending",
    "saving_emergency_funds",
    "financial_planning"
}

PERSONAL_ACCOUNTING_SIGNALS = [
    "personal finance",
    "personal balance sheet",
    "personal budget",
    "household",
    "family budget",
    "our finances",
    "budget plan"
]

In [76]:
def should_auto_exclude(row):
    flags = set(row["exclusion_flags"])
    topics = set(row["matched_topics"])
    question = row["question_clean"].lower()

    # Clearly jurisdiction-specific / out-of-scope
    if "tax_specific" in flags:
        return True

    # Clearly corporate/business finance
    if "corporate_finance" in flags:
        return True

    # Specific securities / market mechanics
    if "specific_securities" in flags:
        return True

    # Accounting:
    # preserve explicitly personal / household finance cases
    if "business_accounting" in flags:
        personal_context = any(
            signal in question
            for signal in PERSONAL_ACCOUNTING_SIGNALS
        )

        if not personal_context:
            return True

    # Advanced trading:
    # protect strong core financial-literacy questions
    # for later review rather than deleting automatically
    if "advanced_trading" in flags:
        if topics.intersection(CORE_TOPICS):
            return False

        return True

    return False

In [77]:
df_candidates["auto_exclude"] = df_candidates.apply(should_auto_exclude, axis=1)

df_relevant_stage1 = df_candidates[~df_candidates["auto_exclude"]].copy()

print("QA pairs auto-excluded:", df_candidates["auto_exclude"].sum())
print("QA pairs remaining:", len(df_relevant_stage1))
print("Unique questions remaining:", df_relevant_stage1["question_clean"].nunique())

QA pairs auto-excluded: 242
QA pairs remaining: 6314
Unique questions remaining: 2094


Let's also add a positive priority score: Instead of treating all topics equally, we can score questions according to how strongly they fit our target.

In [78]:
TOPIC_WEIGHTS = {
    "student_finance": 3,
    "budgeting_spending": 3,
    "saving_emergency_funds": 3,
    "financial_planning": 3,

    "banking": 2,
    "credit_and_debt": 2,
    "income_and_cashflow": 2,
    "financial_safety": 2,

    "insurance_and_risk": 1,
    "basic_investing": 1
}

In [79]:
def topic_priority_score(topics):
    return sum(
        TOPIC_WEIGHTS.get(topic, 0)
        for topic in topics
    )

df_relevant_stage1["priority_score"] = (df_relevant_stage1["matched_topics"].apply(topic_priority_score))

Next, let's inspect the QA-pair distribution and the unique-question distribution.

In [80]:
priority_qa_summary = (
    df_relevant_stage1["priority_score"]
    .value_counts()
    .sort_index()
    .rename_axis("priority_score")
    .reset_index(name="qa_pairs")
)

display(priority_qa_summary)

,priority_score,qa_pairs
0,1,2045
1,2,2386
2,3,1015
3,4,346
4,5,365
5,6,89
6,7,34
7,8,32
8,9,2


In [81]:
priority_question_summary = (
    df_relevant_stage1
    .drop_duplicates("question_clean")
    ["priority_score"]
    .value_counts()
    .sort_index()
    .rename_axis("priority_score")
    .reset_index(name="unique_questions")
)

display(priority_question_summary)

,priority_score,unique_questions
0,1,729
1,2,794
2,3,301
3,4,110
4,5,118
5,6,23
6,7,8
7,8,10
8,9,1


Let's inspect exact topic combinations

In [87]:
def topic_combination(topics):
    return " + ".join(sorted(topics))

df_relevant_stage1["topic_combination"] = (df_relevant_stage1["matched_topics"] .apply(topic_combination))

topic_combination_summary = (
    df_relevant_stage1
    .groupby(
        ["priority_score", "topic_combination"]
    ).agg(
        qa_pairs=("question_clean", "size"),
        unique_questions=("question_clean", "nunique")
    )
    .reset_index().sort_values(
        ["priority_score", "unique_questions"],
        ascending=[True, False]
    ))

display(topic_combination_summary.head(10))

,priority_score,topic_combination,qa_pairs,unique_questions
0,1,basic_investing,1786,643
1,1,insurance_and_risk,259,86
4,2,credit_and_debt,1415,441
2,2,banking,473,179
6,2,income_and_cashflow,369,141
5,2,financial_safety,122,30
3,2,basic_investing + insurance_and_risk,7,3
17,3,saving_emergency_funds,319,93
12,3,budgeting_spending,296,82
14,3,financial_planning,125,38


In [88]:
display(topic_combination_summary.tail(10))

,priority_score,topic_combination,qa_pairs,unique_questions
61,7,basic_investing + saving_emergency_funds + student_finance,10,1
62,7,credit_and_debt + income_and_cashflow + student_finance,2,1
64,8,banking + saving_emergency_funds + student_finance,4,2
66,8,budgeting_spending + credit_and_debt + student_finance,6,2
69,8,credit_and_debt + saving_emergency_funds + student_finance,9,2
63,8,banking + basic_investing + income_and_cashflow + saving_emergency_funds,2,1
65,8,budgeting_spending + credit_and_debt + financial_planning,5,1
67,8,budgeting_spending + financial_planning + income_and_cashflow,3,1
68,8,budgeting_spending + income_and_cashflow + saving_emergency_funds,3,1
70,9,basic_investing + credit_and_debt + saving_emergency_funds + student_finance,2,1


The latest outputs already tell us what we need: **the low-priority pool** is dominated by basic_investing, and many of those questions are about specific ETFs, portfolio theory, real estate investing, individual stocks, and trading mechanics rather than beginner financial literacy. Meanwhile, **the higher-priority** examples contain exactly the material we want: student loans, saving, budgeting, debt repayment, bank accounts, and beginner investing.

In [89]:
JURISDICTION_TERMS = [
    "irs",
    "1040",
    "1099",
    "w-2",
    "w2",
    "schedule c",
    "schedule d",

    "401k",
    "401(k)",
    "roth ira",
    "traditional ira",
    "fha loan",

    "student finance england",
    "parent plus",
    "f-1 visa",

    "hmrc",
    "resp",

    "u.s. tax",
    "us tax",
    "uk tax",
    "canadian tax",
    "canada tax",
    "australian tax",
    "india tax"
]


def is_jurisdiction_specific(question):
    question = question.lower()

    return any(
        term in question
        for term in JURISDICTION_TERMS
    )

In [90]:
df_relevant_stage1["jurisdiction_specific"] = (
    df_relevant_stage1["question_clean"]
    .apply(is_jurisdiction_specific)
)

Next, let'd apply that jurisdiction flag to our already filtered dataset and create the actual final FiQA subset.

In [91]:
df_final_fiqa = df_relevant_stage1[~df_relevant_stage1["jurisdiction_specific"]].copy()

df_final_fiqa.reset_index(drop=True, inplace=True)

print("Rows before jurisdiction filtering:", len(df_relevant_stage1))
print("Jurisdiction-specific rows removed:",
      df_relevant_stage1["jurisdiction_specific"].sum())
print("Rows after jurisdiction filtering:", len(df_final_fiqa))
print("Unique questions remaining:",
      df_final_fiqa["question_clean"].nunique())

Rows before jurisdiction filtering: 6314
Jurisdiction-specific rows removed: 257
Rows after jurisdiction filtering: 6057
Unique questions remaining: 1996


In [92]:
incomplete_answer = (
    df_final_fiqa["answer_clean"].str.endswith(":") &
    (df_final_fiqa["answer_words"] < 30)
)

print("Short incomplete colon-ending answers:", incomplete_answer.sum())

df_final_fiqa = df_final_fiqa[~incomplete_answer].copy()

df_final_fiqa.reset_index(drop=True, inplace=True)

print("Final QA pairs:", len(df_final_fiqa))
print("Final unique questions:", df_final_fiqa["question_clean"].nunique())

Short incomplete colon-ending answers: 38
Final QA pairs: 6019
Final unique questions: 1991


## Now, it's time to save the final FiQA dataset

In [93]:
FINAL_COLUMNS = [
    "question_clean",
    "answer_clean",
    "matched_topics"
]

df_final_fiqa[FINAL_COLUMNS].to_csv("fiqa_student_financial_literacy_clean.csv", index=False)
print("Saved:", len(df_final_fiqa), "QA pairs")

Saved: 6019 QA pairs


Let's now split it by unique questions

In [94]:
RANDOM_STATE = 42

# Get unique questions
unique_questions = (df_final_fiqa["question_clean"].drop_duplicates().to_numpy())

rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(unique_questions)

n_questions = len(unique_questions)

n_train = int(0.80 * n_questions)
n_val = int(0.10 * n_questions)

train_questions = set(unique_questions[:n_train])
val_questions = set(unique_questions[n_train:n_train + n_val])
test_questions = set(unique_questions[n_train + n_val:])

In [95]:
train_df = df_final_fiqa[df_final_fiqa["question_clean"].isin(train_questions)].copy()
val_df = df_final_fiqa[df_final_fiqa["question_clean"].isin(val_questions)].copy()
test_df = df_final_fiqa[df_final_fiqa["question_clean"].isin(test_questions)].copy()

train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [96]:
print("TRAIN")
print("QA pairs:", len(train_df))
print("Unique questions:", train_df["question_clean"].nunique())

print("\nVALIDATION")
print("QA pairs:", len(val_df))
print("Unique questions:", val_df["question_clean"].nunique())

print("\nTEST")
print("QA pairs:", len(test_df))
print("Unique questions:", test_df["question_clean"].nunique())

print("\nTOTAL QA pairs:",
      len(train_df) + len(val_df) + len(test_df))

print("TOTAL unique questions:",
      train_df["question_clean"].nunique()
      + val_df["question_clean"].nunique()
      + test_df["question_clean"].nunique())

TRAIN
QA pairs: 4852
Unique questions: 1592

VALIDATION
QA pairs: 578
Unique questions: 199

TEST
QA pairs: 589
Unique questions: 200

TOTAL QA pairs: 6019
TOTAL unique questions: 1991


Then let's verify there is 0 question leakage

In [97]:
train_q = set(train_df["question_clean"])
val_q = set(val_df["question_clean"])
test_q = set(test_df["question_clean"])

print("Train-Val overlap:", len(train_q & val_q))
print("Train-Test overlap:", len(train_q & test_q))
print("Val-Test overlap:", len(val_q & test_q))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


Finallly, let's save the splits

In [98]:
train_df.to_csv("fiqa_train.csv", index=False)
val_df.to_csv("fiqa_validation.csv", index=False)
test_df.to_csv("fiqa_test.csv", index=False)

print("Train/validation/test datasets saved.")

Train/validation/test datasets saved.
